# 🚢 Tutorial Data Cleaning — Dataset Titanic
### Panduan Lengkap 13 Tahapan Data Cleaning Menggunakan Python

---

Notebook ini merupakan **versi Python (Google Colab)** dari tutorial data cleaning dataset Titanic yang sebelumnya dibuat menggunakan Microsoft Excel.

| Info | Detail |
|------|--------|
| **Total Baris** | 891 |
| **Kolom Awal** | 12 |
| **Tahapan** | 13 (Step 0–12) |
| **Tool Utama** | Python (Pandas, NumPy, Matplotlib, Seaborn) |
| **Sumber Dataset** | [Kaggle: Titanic — Machine Learning from Disaster](https://www.kaggle.com/c/titanic/data) |

## 📦 Install & Import Library

Library yang digunakan:
- **pandas** — Manipulasi dan analisis data
- **numpy** — Operasi numerik
- **matplotlib** & **seaborn** — Visualisasi data
- **re** — Regular expression untuk membersihkan teks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings('ignore')

# Styling untuk visualisasi
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print('✅ Semua library berhasil di-import!')

---
## Step 0 — Persiapan: Membaca File CSV

**Setara Excel:** *Membuka file CSV dan memastikan data terpisah ke kolom dengan benar*

### 💡 Mengapa Ini Penting?
Di Excel, kita perlu menggunakan **Text to Columns** atau **Import Data** untuk memastikan CSV terbaca dengan benar. Di Python, `pandas.read_csv()` secara otomatis menangani pemisah koma, encoding UTF-8, dan teks yang mengandung koma di dalam tanda kutip.

> **Keuntungan Python:** Tidak perlu khawatir tentang pengaturan regional (titik koma vs koma) atau text qualifier — `pd.read_csv()` menangani semuanya secara otomatis!

In [ ]:
# =============================================
# STEP 0: Membaca File CSV
# =============================================

# --- Opsi 1: Upload file dari komputer (Google Colab) ---
# from google.colab import files
# uploaded = files.upload()  # Akan muncul dialog upload
# df = pd.read_csv('Titanic-Dataset.csv')

# --- Opsi 2: Baca langsung dari URL Kaggle/GitHub ---
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# --- Opsi 3: Baca dari file lokal ---
# df = pd.read_csv('Titanic-Dataset.csv')

print(f'✅ Dataset berhasil dimuat!')
print(f'   Jumlah baris : {df.shape[0]}')
print(f'   Jumlah kolom : {df.shape[1]}')
print(f'   Kolom        : {list(df.columns)}')

### 👀 Preview Dataset
Berikut adalah 6 baris pertama dari dataset Titanic. Perhatikan kolom-kolom yang memiliki nilai kosong (NaN).

In [ ]:
# Preview 6 baris pertama (setara melihat data di Excel)
df.head(6)

In [ ]:
# Informasi umum dataset (setara memeriksa tipe kolom di Excel)
print('📊 Informasi Dataset:')
print('=' * 50)
df.info()

In [ ]:
# Statistik deskriptif (setara menggunakan fungsi statistik di Excel)
df.describe()

---
## Step 1 — Memeriksa Missing Values

**Setara Excel:** `=COUNTBLANK(F2:F892)` + Conditional Formatting → Blanks

### 💡 Penjelasan
Missing values (nilai kosong) adalah data yang tidak terisi pada suatu baris. Dalam dataset Titanic, kolom **Age**, **Cabin**, dan **Embarked** memiliki banyak missing values. Langkah pertama data cleaning adalah mengetahui seberapa banyak data yang hilang di setiap kolom agar kita bisa memutuskan penanganan yang tepat.

In [ ]:
# =============================================
# STEP 1: Memeriksa Missing Values
# =============================================
# Setara Excel: =COUNTBLANK(F2:F892)

# Hitung jumlah dan persentase missing values per kolom
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# Buat tabel ringkasan
missing_summary = pd.DataFrame({
    'Kolom': df.columns,
    'Jumlah Kosong': missing_count.values,
    'Persentase (%)': missing_pct.values,
    'Status': ['✅ Lengkap' if x == 0 
               else '❌ Sangat Banyak' if x > 50 
               else '⚠️ Perlu Ditangani' if x > 5 
               else '⚠️ Sedikit' 
               for x in missing_pct.values]
})

print('📋 Hasil Pemeriksaan Missing Values')
print('=' * 60)
print(missing_summary.to_string(index=False))

In [ ]:
# Visualisasi missing values (setara Conditional Formatting di Excel)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart jumlah missing
colors = ['#EF4444' if x > 0 else '#22C55E' for x in missing_count.values]
axes[0].barh(df.columns, missing_count.values, color=colors, edgecolor='#1A1A1A', linewidth=1.5)
axes[0].set_xlabel('Jumlah Missing Values')
axes[0].set_title('Jumlah Missing Values per Kolom', fontweight='bold')
for i, v in enumerate(missing_count.values):
    if v > 0:
        axes[0].text(v + 5, i, str(v), va='center', fontweight='bold', color='#EF4444')

# Heatmap missing values
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Peta Missing Values (Kuning = Ada Data, Merah = Kosong)', fontweight='bold')

plt.tight_layout()
plt.show()

### 📊 Kesimpulan Step 1
| Kolom | Jumlah Kosong | Persentase | Status |
|-------|:---:|:---:|---|
| **Age** | 177 | 19.87% | ⚠️ Perlu Ditangani |
| **Cabin** | 687 | 77.10% | ❌ Sangat Banyak |
| **Embarked** | 2 | 0.22% | ⚠️ Sedikit |

---
## Step 2 — Menangani Missing Values

**Setara Excel:**
- `=MEDIAN(F2:F892)` untuk menghitung median Age
- `=INDEX(...MATCH(MAX(COUNTIF(...))))` untuk mencari modus Embarked
- Klik kanan header kolom Cabin → Delete

### 💡 Penjelasan
Strategi penanganan missing values:
- **Age** (19.87% kosong): Isi dengan **median** — karena data numerik dan bisa memiliki outlier
- **Embarked** (0.22% kosong): Isi dengan **modus** (nilai paling sering muncul) — karena data kategorikal
- **Cabin** (77.10% kosong): **Hapus kolom** — terlalu banyak data hilang

> **Tips:** Median lebih disarankan daripada rata-rata (AVERAGE) karena tidak terpengaruh oleh outlier.

In [ ]:
# =============================================
# STEP 2: Menangani Missing Values
# =============================================

# --- 2a. Mengisi Age dengan Median ---
# Setara Excel: =MEDIAN(F2:F892) → 28, lalu Find & Replace sel kosong dengan 28
median_age = df['Age'].median()
print(f'📌 Median Age: {median_age}')

df['Age'].fillna(median_age, inplace=True)
print(f'   ✅ {177} missing values pada Age diisi dengan median ({median_age})')

# --- 2b. Mengisi Embarked dengan Modus ---
# Setara Excel: =INDEX(L2:L892,MATCH(MAX(COUNTIF(L2:L892,L2:L892)),COUNTIF(L2:L892,L2:L892),0))
mode_embarked = df['Embarked'].mode()[0]
print(f'\n📌 Modus Embarked: "{mode_embarked}" (Southampton)')

df['Embarked'].fillna(mode_embarked, inplace=True)
print(f'   ✅ 2 missing values pada Embarked diisi dengan modus ("{mode_embarked}")')

# --- 2c. Menghapus Kolom Cabin ---
# Setara Excel: Klik kanan header kolom K → Delete
df.drop(columns=['Cabin'], inplace=True)
print(f'\n🗑️  Kolom Cabin dihapus (77.10% data kosong — tidak berguna untuk analisis)')

# Verifikasi: Cek ulang missing values
print(f'\n📋 Verifikasi — Missing values tersisa:')
remaining_missing = df.isnull().sum()
print(remaining_missing[remaining_missing > 0] if remaining_missing.sum() > 0 else '   ✅ Tidak ada missing values lagi!')

---
## Step 3 — Memeriksa Data Duplikat

**Setara Excel:** Conditional Formatting → Highlight Cells Rules → Duplicate Values + kolom helper COUNTIF

### 💡 Penjelasan
Data duplikat adalah baris yang memiliki nilai identik di semua kolom. Duplikat dapat mengakibatkan hasil analisis menjadi bias karena data dihitung lebih dari sekali. Pada dataset Titanic, setiap penumpang seharusnya unik berdasarkan **PassengerId**.

In [ ]:
# =============================================
# STEP 3: Memeriksa Data Duplikat
# =============================================
# Setara Excel: Conditional Formatting → Duplicate Values
# Setara Excel: =COUNTIF(M:M,M2) > 1

# Cek duplikat berdasarkan SEMUA kolom
jumlah_duplikat_semua = df.duplicated().sum()
print(f'📋 Hasil Pemeriksaan Duplikat')
print('=' * 50)
print(f'   Duplikat (semua kolom)     : {jumlah_duplikat_semua}')

# Cek duplikat berdasarkan PassengerId saja
jumlah_duplikat_id = df.duplicated(subset=['PassengerId']).sum()
print(f'   Duplikat (PassengerId saja): {jumlah_duplikat_id}')

if jumlah_duplikat_semua == 0:
    print(f'\n   ✅ Tidak ditemukan data duplikat!')
    print(f'   ℹ️  Setiap baris memiliki PassengerId unik — sesuai ekspektasi.')
else:
    print(f'\n   ⚠️ Ditemukan {jumlah_duplikat_semua} baris duplikat!')
    print('   Baris duplikat:')
    print(df[df.duplicated(keep=False)])

---
## Step 4 — Menghapus Data Duplikat

**Setara Excel:** Tab Data → Remove Duplicates → Select All → OK

### 💡 Penjelasan
Jika pada langkah sebelumnya ditemukan data duplikat, langkah ini bertujuan menghapusnya. Pada dataset Titanic umumnya tidak ada duplikat, tetapi tetap penting mengetahui caranya.

> ⚠️ **Perhatian:** Di Excel, Remove Duplicates bersifat permanen setelah file disimpan. Di Python, kita bisa selalu memuat ulang data dari file asli.

In [ ]:
# =============================================
# STEP 4: Menghapus Data Duplikat
# =============================================
# Setara Excel: Data → Remove Duplicates → Select All → OK

jumlah_sebelum = len(df)

# Hapus duplikat (keep='first' menyimpan kemunculan pertama)
df.drop_duplicates(inplace=True)

jumlah_sesudah = len(df)
jumlah_dihapus = jumlah_sebelum - jumlah_sesudah

print(f'📋 Hasil Penghapusan Duplikat')
print('=' * 50)
print(f'   Jumlah baris sebelum : {jumlah_sebelum}')
print(f'   Jumlah baris sesudah : {jumlah_sesudah}')
print(f'   Baris dihapus        : {jumlah_dihapus}')

if jumlah_dihapus == 0:
    print(f'\n   ✅ Tidak ada duplikat yang dihapus (data sudah unik).')
else:
    print(f'\n   🗑️ {jumlah_dihapus} baris duplikat berhasil dihapus.')

---
## Step 5 — Memeriksa Tipe Data

**Setara Excel:** `=ISNUMBER(F2)`, `=ISTEXT(E2)`, `=TYPE(F2)`, `=SUMPRODUCT((ISTEXT(F2:F892))*1)`

### 💡 Penjelasan
Tipe data yang salah bisa menyebabkan error dalam perhitungan. Kita perlu memastikan:
- **Age** dan **Fare** → Numerik (float)
- **Survived** dan **Pclass** → Numerik/Kategori (int)
- **Name**, **Sex**, **Embarked** → Teks (object/string)

In [ ]:
# =============================================
# STEP 5: Memeriksa Tipe Data
# =============================================
# Setara Excel: =ISNUMBER(F2), =ISTEXT(E2), =TYPE(F2)

# Tipe data yang diharapkan
expected_types = {
    'PassengerId': 'Integer (ID unik)',
    'Survived': 'Integer (0/1 — Biner)',
    'Pclass': 'Integer (1/2/3 — Kelas tiket)',
    'Name': 'Text (Nama lengkap)',
    'Sex': 'Text (Kategorikal)',
    'Age': 'Float (Usia dalam tahun)',
    'SibSp': 'Integer (Saudara/pasangan)',
    'Parch': 'Integer (Orangtua/anak)',
    'Ticket': 'Text (Nomor tiket)',
    'Fare': 'Float (Harga tiket)',
    'Embarked': 'Text (Pelabuhan)'
}

print('📋 Pemeriksaan Tipe Data')
print('=' * 70)
print(f'{"Kolom":<15} {"Tipe Aktual":<15} {"Tipe Diharapkan":<35} {"Status"}')
print('-' * 70)

for col in df.columns:
    actual = str(df[col].dtype)
    expected = expected_types.get(col, '-')
    # Cek apakah tipe sudah sesuai
    is_ok = True
    if col in ['Age', 'Fare'] and 'float' not in actual and 'int' not in actual:
        is_ok = False
    elif col in ['PassengerId', 'Survived', 'Pclass', 'SibSp', 'Parch'] and 'int' not in actual:
        is_ok = False
    elif col in ['Name', 'Sex', 'Ticket', 'Embarked'] and actual != 'object':
        is_ok = False
    status = '✅' if is_ok else '⚠️ Perlu Konversi'
    print(f'{col:<15} {actual:<15} {expected:<35} {status}')

In [ ]:
# Konversi tipe data jika diperlukan
# Setara Excel: VALUE() untuk teks→angka, TEXT() untuk angka→teks

# Pastikan kolom integer benar-benar integer
for col in ['PassengerId', 'Survived', 'Pclass', 'SibSp', 'Parch']:
    df[col] = df[col].astype(int)

# Pastikan kolom float benar-benar float
for col in ['Age', 'Fare']:
    df[col] = df[col].astype(float)

print('✅ Tipe data sudah disesuaikan!')
print()
print(df.dtypes)

---
## Step 6 — Menyeragamkan Format Data

**Setara Excel:**
- `=TRIM(CLEAN(D2))` — Bersihkan spasi & karakter non-cetak
- `=LOWER(TRIM(E2))` — Huruf kecil semua
- `=UPPER(TRIM(L2))` — Huruf besar semua
- Copy hasil → Paste Values

### 💡 Penjelasan
Data dari berbagai sumber sering memiliki format yang tidak konsisten: spasi berlebih, huruf besar/kecil yang tidak seragam, atau karakter tidak terlihat. Ini terutama relevan untuk kolom **Name**, **Sex**, dan **Embarked**.

In [ ]:
# =============================================
# STEP 6: Menyeragamkan Format Data
# =============================================

print('📋 Sebelum Pembersihan Format:')
print(f'   Nilai unik Sex     : {df["Sex"].unique()}')
print(f'   Nilai unik Embarked: {df["Embarked"].unique()}')

# --- 6a. Bersihkan spasi berlebih (setara TRIM + CLEAN di Excel) ---
for col in ['Name', 'Sex', 'Ticket', 'Embarked']:
    df[col] = df[col].str.strip()  # Hapus spasi di awal/akhir (TRIM)
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)  # Hapus spasi ganda

# --- 6b. Seragamkan huruf kecil pada Sex (setara LOWER di Excel) ---
df['Sex'] = df['Sex'].str.lower()

# --- 6c. Seragamkan huruf besar pada Embarked (setara UPPER di Excel) ---
df['Embarked'] = df['Embarked'].str.upper()

print(f'\n📋 Sesudah Pembersihan Format:')
print(f'   Nilai unik Sex     : {df["Sex"].unique()}')
print(f'   Nilai unik Embarked: {df["Embarked"].unique()}')
print(f'\n   ✅ Format data sudah diseragamkan!')

---
## Step 7 — Membersihkan Karakter Khusus

**Setara Excel:**
- `=SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(I2,"/",""),".","")," ","")` — Hapus karakter khusus
- `=TEXTJOIN("",TRUE,IF(ISNUMBER(...)))` — Ekstrak hanya angka
- Find & Replace (Ctrl+H) — Hapus karakter tertentu

### 💡 Penjelasan
Kolom **Ticket** memiliki campuran huruf, angka, dan simbol (misalnya "A/5 21171", "STON/O2. 3101282"). Jika ingin menganalisis nomor tiket, karakter ini perlu dibersihkan.

In [ ]:
# =============================================
# STEP 7: Membersihkan Karakter Khusus
# =============================================

print('📋 Contoh Ticket Sebelum Pembersihan:')
print(df['Ticket'].head(10).to_string())

# --- 7a. Hapus karakter khusus dari Ticket ---
# Setara Excel: =SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(I2,"/",""),".","")," ","")
df['Ticket_Clean'] = df['Ticket'].str.replace(r'[/. ]', '', regex=True)

print(f'\n📋 Contoh Ticket Sesudah Pembersihan (hapus /, titik, spasi):')
print(pd.DataFrame({
    'Ticket_Asli': df['Ticket'].head(10).values,
    'Ticket_Clean': df['Ticket_Clean'].head(10).values
}).to_string(index=False))

# --- 7b. Ekstrak hanya angka dari Ticket ---
# Setara Excel: =TEXTJOIN("",TRUE,IF(ISNUMBER(MID(...)*1),MID(...),""))
df['Ticket_NumOnly'] = df['Ticket'].str.replace(r'[^0-9]', '', regex=True)

print(f'\n📋 Contoh Ticket (angka saja):')
print(pd.DataFrame({
    'Ticket_Asli': df['Ticket'].head(10).values,
    'Ticket_NumOnly': df['Ticket_NumOnly'].head(10).values
}).to_string(index=False))

# Hapus kolom bantuan (kita simpan Ticket asli)
df.drop(columns=['Ticket_Clean', 'Ticket_NumOnly'], inplace=True)
print(f'\n   ✅ Demonstrasi pembersihan karakter khusus selesai.')
print(f'   ℹ️  Kolom Ticket asli dipertahankan (kolom bantuan dihapus).')

---
## Step 8 — Menangani Outlier

**Setara Excel:**
- `=QUARTILE(J2:J892,1)` → Q1
- `=QUARTILE(J2:J892,3)` → Q3
- IQR = Q3 - Q1
- `=IF(OR(J2<Q1-1.5*IQR, J2>Q3+1.5*IQR),"OUTLIER","Normal")`
- Boxplot via Insert → Chart → Box and Whisker

### 💡 Penjelasan
Outlier adalah nilai yang jauh berbeda dari mayoritas data. Pada dataset Titanic, kolom **Fare** memiliki outlier sangat mencolok — beberapa penumpang membayar harga tiket sangat tinggi (misalnya 512.33) dibanding rata-rata. Metode **IQR (Interquartile Range)** digunakan untuk mendeteksi outlier secara statistik.

> **Tips:** Pada kasus Titanic, outlier Fare mungkin memang valid (penumpang kelas 1 membayar sangat mahal). Pertimbangkan konteks sebelum menghapus outlier.

In [ ]:
# =============================================
# STEP 8: Menangani Outlier
# =============================================

# --- 8a. Hitung Q1, Q3, IQR untuk Fare ---
# Setara Excel: =QUARTILE(J2:J892,1) dan =QUARTILE(J2:J892,3)
Q1_fare = df['Fare'].quantile(0.25)
Q3_fare = df['Fare'].quantile(0.75)
IQR_fare = Q3_fare - Q1_fare

batas_bawah = Q1_fare - 1.5 * IQR_fare
batas_atas = Q3_fare + 1.5 * IQR_fare

print('📊 Analisis Outlier — Kolom Fare')
print('=' * 50)
print(f'   Q1 (Kuartil 1) : {Q1_fare:.2f}')
print(f'   Q3 (Kuartil 3) : {Q3_fare:.2f}')
print(f'   IQR            : {IQR_fare:.2f}')
print(f'   Batas Bawah    : {batas_bawah:.2f} (atau 0, karena fare tidak bisa negatif)')
print(f'   Batas Atas     : {batas_atas:.2f}')

# --- 8b. Tandai outlier ---
# Setara Excel: =IF(OR(J2<Q1-1.5*IQR, J2>Q3+1.5*IQR),"OUTLIER","Normal")
df['Fare_Status'] = np.where(
    (df['Fare'] < batas_bawah) | (df['Fare'] > batas_atas),
    'OUTLIER', 'Normal'
)

jumlah_outlier = (df['Fare_Status'] == 'OUTLIER').sum()
print(f'\n   Jumlah Outlier : {jumlah_outlier} dari {len(df)} baris')
print(f'   Persentase     : {jumlah_outlier/len(df)*100:.2f}%')

# Tampilkan beberapa contoh outlier
print(f'\n📋 Contoh Outlier Fare:')
print(df[df['Fare_Status'] == 'OUTLIER'][['Name', 'Pclass', 'Fare', 'Fare_Status']].head(10).to_string(index=False))

In [ ]:
# --- 8c. Visualisasi Boxplot ---
# Setara Excel: Insert → Chart → Box and Whisker

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot Fare
bp = axes[0].boxplot(df['Fare'], vert=True, patch_artist=True,
                     boxprops=dict(facecolor='#FACC15', edgecolor='#1A1A1A', linewidth=2),
                     whiskerprops=dict(color='#1A1A1A', linewidth=2),
                     capprops=dict(color='#1A1A1A', linewidth=2),
                     medianprops=dict(color='#EF4444', linewidth=2),
                     flierprops=dict(marker='o', markerfacecolor='#EF4444', markersize=5))
axes[0].set_title('Boxplot Fare — Deteksi Outlier', fontweight='bold')
axes[0].set_ylabel('Fare ($)')
axes[0].axhline(y=batas_atas, color='#EF4444', linestyle='--', label=f'Batas Atas ({batas_atas:.2f})')
axes[0].legend()

# Histogram Fare
axes[1].hist(df['Fare'], bins=50, color='#FACC15', edgecolor='#1A1A1A', linewidth=1)
axes[1].axvline(x=batas_atas, color='#EF4444', linestyle='--', linewidth=2, label=f'Batas Atas ({batas_atas:.2f})')
axes[1].set_title('Distribusi Fare', fontweight='bold')
axes[1].set_xlabel('Fare ($)')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()

# Hapus kolom bantuan
df.drop(columns=['Fare_Status'], inplace=True)
print('\nℹ️  Outlier Fare TIDAK dihapus — karena pada konteks Titanic,')
print('   penumpang kelas 1 memang membayar sangat mahal (data valid).')

---
## Step 9 — Memvalidasi Rentang Nilai

**Setara Excel:**
- `=IF(AND(F2>=0,F2<=120),"Valid","INVALID")` — Validasi Age
- `=IF(OR(C2=1,C2=2,C2=3),"Valid","INVALID")` — Validasi Pclass
- `=IF(OR(B2=0,B2=1),"Valid","INVALID")` — Validasi Survived
- `=IF(J2>=0,"Valid","INVALID")` — Validasi Fare
- `=COUNTIF(M2:M892,"INVALID")` — Hitung total invalid

### 💡 Penjelasan
Validasi memastikan data masuk akal secara logis:
- **Age**: 0–120 tahun
- **Pclass**: Hanya 1, 2, atau 3
- **Survived**: Hanya 0 atau 1
- **Fare**: Tidak negatif

In [ ]:
# =============================================
# STEP 9: Memvalidasi Rentang Nilai
# =============================================

# Definisikan aturan validasi
validations = {
    'Age': {
        'rule': (df['Age'] >= 0) & (df['Age'] <= 120),
        'desc': '0 ≤ Age ≤ 120'
    },
    'Pclass': {
        'rule': df['Pclass'].isin([1, 2, 3]),
        'desc': 'Pclass ∈ {1, 2, 3}'
    },
    'Survived': {
        'rule': df['Survived'].isin([0, 1]),
        'desc': 'Survived ∈ {0, 1}'
    },
    'Fare': {
        'rule': df['Fare'] >= 0,
        'desc': 'Fare ≥ 0'
    },
    'SibSp': {
        'rule': df['SibSp'] >= 0,
        'desc': 'SibSp ≥ 0'
    },
    'Parch': {
        'rule': df['Parch'] >= 0,
        'desc': 'Parch ≥ 0'
    }
}

print('📋 Hasil Validasi Rentang Nilai')
print('=' * 65)
print(f'{"Kolom":<12} {"Aturan":<25} {"Valid":<8} {"Invalid":<8} {"Status"}')
print('-' * 65)

total_invalid = 0
for col, val in validations.items():
    valid_count = val['rule'].sum()
    invalid_count = (~val['rule']).sum()
    total_invalid += invalid_count
    status = '✅' if invalid_count == 0 else '⚠️ Perlu Perbaikan'
    print(f'{col:<12} {val["desc"]:<25} {valid_count:<8} {invalid_count:<8} {status}')

print('-' * 65)
if total_invalid == 0:
    print(f'\n✅ Semua nilai berada dalam rentang yang valid!')
else:
    print(f'\n⚠️ Total data invalid: {total_invalid}')

In [ ]:
# Ringkasan statistik untuk verifikasi rentang
print('📊 Ringkasan Statistik Kolom Numerik')
print('=' * 60)
print(df[['Age', 'Fare', 'SibSp', 'Parch', 'Survived', 'Pclass']].describe().round(2))

---
## Step 10 — Mengubah Data Kategorikal

**Setara Excel:**
- **Label Encoding:** `=IF(E2="male",1,0)` — Sex
- **One-Hot Encoding:** `=IF(L2="S",1,0)`, `=IF(L2="C",1,0)`, `=IF(L2="Q",1,0)` — Embarked

### 💡 Penjelasan
Banyak algoritma machine learning memerlukan input numerik. Data kategorikal seperti **Sex** dan **Embarked** perlu diubah menjadi angka:
- **Label Encoding**: Mengubah kategori menjadi angka (misal: male=1, female=0)
- **One-Hot Encoding**: Membuat kolom biner baru untuk setiap kategori

In [ ]:
# =============================================
# STEP 10: Mengubah Data Kategorikal
# =============================================

# --- 10a. Label Encoding — Sex ---
# Setara Excel: =IF(E2="male",1,0)
df['Sex_Encoded'] = df['Sex'].map({'male': 1, 'female': 0})

print('📋 Label Encoding — Sex')
print('   male   → 1')
print('   female → 0')

# --- 10b. One-Hot Encoding — Embarked ---
# Setara Excel: =IF(L2="S",1,0), =IF(L2="C",1,0), =IF(L2="Q",1,0)
df['Embarked_S'] = (df['Embarked'] == 'S').astype(int)
df['Embarked_C'] = (df['Embarked'] == 'C').astype(int)
df['Embarked_Q'] = (df['Embarked'] == 'Q').astype(int)

print('\n📋 One-Hot Encoding — Embarked')
print('   S → Embarked_S=1, Embarked_C=0, Embarked_Q=0')
print('   C → Embarked_S=0, Embarked_C=1, Embarked_Q=0')
print('   Q → Embarked_S=0, Embarked_C=0, Embarked_Q=1')

# Tampilkan contoh hasil
print('\n📋 Contoh Hasil Encoding:')
print(df[['Sex', 'Sex_Encoded', 'Embarked', 'Embarked_S', 'Embarked_C', 'Embarked_Q']].head(10).to_string(index=False))

In [ ]:
# Alternatif: Menggunakan pd.get_dummies() — cara cepat one-hot encoding
print('📋 Alternatif: pd.get_dummies() (cara cepat):')
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked')
print(embarked_dummies.head(10))
print('\n   ℹ️  pd.get_dummies() otomatis membuat kolom biner untuk setiap kategori.')

---
## Step 11 — Menghapus Kolom Tidak Relevan

**Setara Excel:** Klik kanan header kolom → Delete (untuk PassengerId, Name, Ticket)

### 💡 Penjelasan
Tidak semua kolom berguna untuk analisis atau prediksi:
- **PassengerId**: Hanya ID unik, tidak memiliki makna prediktif
- **Ticket**: Terlalu banyak variasi unik, sulit dikategorikan
- **Name**: Teks unik (kecuali jika diekstrak title-nya — lihat Step 12)
- **Cabin**: Sudah dihapus di Step 2

Kolom asli **Sex** dan **Embarked** juga bisa dihapus karena sudah di-encode di Step 10.

In [ ]:
# =============================================
# STEP 11: Menghapus Kolom Tidak Relevan
# =============================================

kolom_sebelum = list(df.columns)
print(f'📋 Kolom Sebelum ({len(kolom_sebelum)} kolom):')
print(f'   {kolom_sebelum}')

# Kolom yang akan dihapus
kolom_hapus = ['PassengerId', 'Name', 'Ticket', 'Sex', 'Embarked']
print(f'\n🗑️  Kolom yang dihapus: {kolom_hapus}')
print(f'   - PassengerId : Hanya ID unik, tidak prediktif')
print(f'   - Name        : Teks unik (title diekstrak di Step 12)')
print(f'   - Ticket      : Terlalu banyak variasi unik')
print(f'   - Sex         : Sudah di-encode → Sex_Encoded')
print(f'   - Embarked    : Sudah di-encode → Embarked_S/C/Q')

# Simpan kolom Name sebelum dihapus (untuk Step 12)
name_backup = df['Name'].copy() if 'Name' in df.columns else None

df.drop(columns=kolom_hapus, inplace=True, errors='ignore')

kolom_sesudah = list(df.columns)
print(f'\n📋 Kolom Sesudah ({len(kolom_sesudah)} kolom):')
print(f'   {kolom_sesudah}')

print(f'\n   ✅ {len(kolom_sebelum) - len(kolom_sesudah)} kolom berhasil dihapus.')

In [ ]:
# Tampilkan dataset setelah penghapusan kolom
print('📋 Preview Dataset Setelah Penghapusan Kolom:')
df.head(10)

---
## Step 12 — Feature Engineering

**Setara Excel:**
- `=G2+H2+1` — FamilySize
- `=IF(G2+H2=0,1,0)` — IsAlone
- `=MID(D2,FIND(",",D2)+2,FIND(".",D2)-FIND(",",D2)-2)` — Title
- `=IF(F2<=12,"Anak",IF(F2<=18,"Remaja",IF(F2<=60,"Dewasa","Lansia")))` — AgeGroup

### 💡 Penjelasan
Feature engineering adalah proses membuat kolom/fitur baru dari data yang sudah ada untuk meningkatkan kualitas analisis atau akurasi model prediksi:
- **FamilySize**: SibSp + Parch + 1 (diri sendiri)
- **IsAlone**: 1 jika sendirian (FamilySize = 1), 0 jika bersama keluarga
- **Title**: Gelar dari kolom Name (Mr, Mrs, Miss, Master)
- **AgeGroup**: Kategori usia (Anak, Remaja, Dewasa, Lansia)

In [ ]:
# =============================================
# STEP 12: Feature Engineering
# =============================================

# --- 12a. FamilySize ---
# Setara Excel: =G2+H2+1
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
print('✅ Fitur FamilySize dibuat: SibSp + Parch + 1')

# --- 12b. IsAlone ---
# Setara Excel: =IF(G2+H2=0,1,0)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
print('✅ Fitur IsAlone dibuat: 1 jika sendirian, 0 jika bersama keluarga')

# --- 12c. Title (Gelar dari Name) ---
# Setara Excel: =MID(D2,FIND(",",D2)+2,FIND(".",D2)-FIND(",",D2)-2)
if name_backup is not None:
    df['Title'] = name_backup.str.extract(r', ([A-Za-z]+)\.', expand=False)
    
    # Kelompokkan title yang jarang muncul
    title_counts = df['Title'].value_counts()
    print(f'\n📋 Distribusi Title:')
    print(title_counts)
    
    # Ganti title langka dengan "Rare"
    rare_titles = title_counts[title_counts < 10].index
    df['Title'] = df['Title'].replace(rare_titles, 'Rare')
    # Seragamkan beberapa title
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    print(f'\n✅ Fitur Title dibuat dan title langka dikelompokkan → "Rare"')
    print(f'   Title final: {df["Title"].unique()}')
else:
    print('⚠️ Kolom Name tidak tersedia — Title tidak bisa diekstrak.')

# --- 12d. AgeGroup ---
# Setara Excel: =IF(F2<=12,"Anak",IF(F2<=18,"Remaja",IF(F2<=60,"Dewasa","Lansia")))
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[0, 12, 18, 60, 120],
    labels=['Anak', 'Remaja', 'Dewasa', 'Lansia']
)
print(f'\n✅ Fitur AgeGroup dibuat:')
print(f'   0-12  → Anak')
print(f'   13-18 → Remaja')
print(f'   19-60 → Dewasa')
print(f'   61+   → Lansia')

In [ ]:
# Tampilkan contoh hasil feature engineering
print('📋 Contoh Hasil Feature Engineering:')
df[['SibSp', 'Parch', 'Age', 'FamilySize', 'IsAlone', 'Title', 'AgeGroup']].head(10)

In [ ]:
# Visualisasi fitur baru
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# FamilySize distribution
df['FamilySize'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0, 0], color='#FACC15', edgecolor='#1A1A1A', linewidth=1.5)
axes[0, 0].set_title('Distribusi FamilySize', fontweight='bold')
axes[0, 0].set_xlabel('FamilySize')
axes[0, 0].set_ylabel('Jumlah')

# IsAlone
alone_counts = df['IsAlone'].value_counts()
axes[0, 1].pie(alone_counts, labels=['Bersama Keluarga', 'Sendirian'],
               autopct='%1.1f%%', colors=['#FACC15', '#3B82F6'],
               wedgeprops=dict(edgecolor='#1A1A1A', linewidth=2))
axes[0, 1].set_title('Proporsi IsAlone', fontweight='bold')

# Title distribution
if 'Title' in df.columns:
    df['Title'].value_counts().plot(
        kind='bar', ax=axes[1, 0], color='#22C55E', edgecolor='#1A1A1A', linewidth=1.5)
    axes[1, 0].set_title('Distribusi Title', fontweight='bold')
    axes[1, 0].set_xlabel('Title')
    axes[1, 0].set_ylabel('Jumlah')
    axes[1, 0].tick_params(axis='x', rotation=45)

# AgeGroup distribution
df['AgeGroup'].value_counts().plot(
    kind='bar', ax=axes[1, 1], color='#F59E0B', edgecolor='#1A1A1A', linewidth=1.5)
axes[1, 1].set_title('Distribusi AgeGroup', fontweight='bold')
axes[1, 1].set_xlabel('AgeGroup')
axes[1, 1].set_ylabel('Jumlah')

plt.tight_layout()
plt.show()

---
## 📊 Ringkasan Final Dataset

Setelah menyelesaikan seluruh 13 tahapan data cleaning, berikut adalah kondisi akhir dataset:

In [ ]:
# =============================================
# RINGKASAN FINAL
# =============================================

print('🏁 RINGKASAN FINAL — Dataset Titanic Setelah Data Cleaning')
print('=' * 60)
print(f'   Jumlah baris  : {df.shape[0]}')
print(f'   Jumlah kolom  : {df.shape[1]}')
print(f'   Missing values: {df.isnull().sum().sum()}')
print(f'   Duplikat      : {df.duplicated().sum()}')
print(f'\n📋 Kolom Final:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:2d}. {col:<15} ({df[col].dtype})')

print('\n📋 Tahapan yang Telah Dilakukan:')
tahapan = [
    'Step 0  : Membaca file CSV (pd.read_csv)',
    'Step 1  : Memeriksa missing values (isnull)',
    'Step 2  : Menangani missing values (fillna, drop)',
    'Step 3  : Memeriksa data duplikat (duplicated)',
    'Step 4  : Menghapus data duplikat (drop_duplicates)',
    'Step 5  : Memeriksa tipe data (dtypes, astype)',
    'Step 6  : Menyeragamkan format data (str.strip, str.lower)',
    'Step 7  : Membersihkan karakter khusus (str.replace, regex)',
    'Step 8  : Menangani outlier (IQR, quantile, boxplot)',
    'Step 9  : Memvalidasi rentang nilai (conditional checks)',
    'Step 10 : Mengubah data kategorikal (label/one-hot encoding)',
    'Step 11 : Menghapus kolom tidak relevan (drop columns)',
    'Step 12 : Feature engineering (FamilySize, IsAlone, Title, AgeGroup)',
]
for t in tahapan:
    print(f'   ✅ {t}')

In [ ]:
# Preview dataset final
print('📋 Preview Dataset Final (10 baris pertama):')
df.head(10)

In [ ]:
# Informasi lengkap dataset final
print('📊 Info Dataset Final:')
print('=' * 50)
df.info()

In [ ]:
# Statistik deskriptif final
print('📊 Statistik Deskriptif Final:')
df.describe()

In [ ]:
# =============================================
# SIMPAN DATASET BERSIH
# =============================================

# Simpan ke file CSV
output_filename = 'Titanic_Cleaned.csv'
df.to_csv(output_filename, index=False)
print(f'💾 Dataset bersih disimpan ke: {output_filename}')
print(f'   Ukuran: {df.shape[0]} baris × {df.shape[1]} kolom')

# Download file (Google Colab)
# from google.colab import files
# files.download(output_filename)

---

## 📚 Perbandingan Excel vs Python

| Tahapan | Excel | Python |
|---------|-------|--------|
| **Buka CSV** | Data → From Text/CSV, Text to Columns | `pd.read_csv()` |
| **Missing Values** | `=COUNTBLANK()` | `df.isnull().sum()` |
| **Isi Missing** | `=MEDIAN()`, Find & Replace | `df.fillna()` |
| **Cek Duplikat** | Conditional Formatting, `=COUNTIF()` | `df.duplicated()` |
| **Hapus Duplikat** | Data → Remove Duplicates | `df.drop_duplicates()` |
| **Tipe Data** | `=ISNUMBER()`, `=ISTEXT()`, `=TYPE()` | `df.dtypes`, `df.astype()` |
| **Format Data** | `=TRIM()`, `=CLEAN()`, `=LOWER()` | `str.strip()`, `str.lower()` |
| **Karakter Khusus** | `=SUBSTITUTE()`, Find & Replace | `str.replace(regex)` |
| **Outlier** | `=QUARTILE()`, Boxplot manual | `quantile()`, `sns.boxplot()` |
| **Validasi** | `=IF(AND())`, `=COUNTIF("INVALID")` | Boolean indexing |
| **Encoding** | `=IF(E2="male",1,0)` | `map()`, `get_dummies()` |
| **Hapus Kolom** | Klik kanan → Delete | `df.drop(columns=[])` |
| **Feature Eng.** | `=MID()`, `=FIND()`, `=IF()` | `str.extract()`, `pd.cut()` |

---

### 🎓 Sumber Dataset
Dataset resmi Titanic dari kompetisi Kaggle: [Machine Learning from Disaster](https://www.kaggle.com/c/titanic/data)

---
*Tutorial Data Cleaning Dataset Titanic — Versi Python (Google Colab)*